# Model Monitoring

This lecture will mostly focus on a free tool called **ML Flow**. This tool is a standard in the industry and is a central part of an ecosystem of monitoring tools.

ML Flow can be used for many tasks, such as deploying and serving models; however, in the scope of this lecture, we will be using this ability to monitor and evaluate models.

## Let's build a model
(take from the lecture "Method Behind The Madness"

In [ ]:
import mlflow
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


import matplotlib
import matplotlib.pyplot as plt
#%matplotlib inline
matplotlib.use('Agg')  # Prevents GUI blocking in background evaluation

In [ ]:
#Load a simple dataset
import seaborn as sns
titanic_df = sns.load_dataset('titanic')

#only keep numeric columns (for simplicity)
titanic_onlynum_df = titanic_df.drop(['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive'], axis=1)

#remove na, nan, etc.
titanic_onlynum_noempty_df = titanic_onlynum_df.dropna()

#### Set up test/train

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    titanic_onlynum_noempty_df.drop(['survived'], axis=1)
    , titanic_onlynum_noempty_df.survived
    , random_state=1)

We change the types to `float` to avoid later problems with model evaluations

In [ ]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)

#### Try different models

Decision Tree

In [ ]:
%%time
from sklearn import tree

model = tree.DecisionTreeClassifier(splitter='best', criterion='gini')
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
model.score(X_test, y_test)

Random Forest

In [ ]:
%%time
from sklearn import ensemble

model = ensemble.RandomForestClassifier(n_estimators=100, min_samples_split=2)
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
model.score(X_test, y_test)

SVM

In [ ]:
%%time
from sklearn import svm

model = svm.SVC(C=1.0, kernel='rbf')
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
model.score(X_test, y_test)

#### What if you had done these experiments across many days or weeks, overwriting old models with new ones?

Doesn't it make sense to learn from scientists and track your experiments?

<img src="images/marie-curie-notebook-1-1.jpg" width="600">
Marie Curie's notebook where she tracked experiments. Still radio active! Credit goes to a great essay at https://press.asimov.com/articles/lab-notebooks

### "Just log, I don't care about the details"

In [ ]:
mlflow.autolog()   # every .fit() below is now recorded automatically

# We are explicitely providing the sqlite argument because ML Flow is transitioning from using local files to sqlite
#mlflow.set_tracking_uri("sqlite:///mlflow.db") 

#mlflow.set_experiment("no-work-experiment")

**BTW** `mlflow.autolog()` also works with `XGBoost`, `Pytorch`, `statsmodels`, `prophet` and other libraries

#### Start the Ml Flow dashboard
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001
```

View it at http://127.0.0.1:5001/

Notice that we started the server on port 5001, instead of the default port of 5000. Macs use port 5000 for a different service so this keeps us from conflicting.

In [ ]:
# Decision Tree
model = tree.DecisionTreeClassifier(splitter='best', criterion='gini')
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
print("Decision Tree:", model.score(X_test, y_test))

# Random Forest
model = ensemble.RandomForestClassifier(n_estimators=100, min_samples_split=2)
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
print("Random Forest:", model.score(X_test, y_test))

# SVM
model = svm.SVC(C=1.0, kernel='rbf')
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
print("SVM:", model.score(X_test, y_test))


Without any effort on your part (and not even running a server), so much information is recorded about each `.fit()`!

In [ ]:
mlflow.search_runs().T      # one row already here — params + metrics you never typed

Note that, along with this data, ML Flow also stores the actual model

<img src="images/mlflow_nowork.png">

Notice that our three models have been automatically stored. But the names are auto-generated.

Visit the dashboard and see what else is stored.

### Let's do proper logging

In [ ]:
mlflow.set_experiment("proper-name-experiment")                 # a named folder for related runs

#### Use `with mlflow.start_run():` to create a new run

In [ ]:
# Decision Tree
model = tree.DecisionTreeClassifier(splitter='best', criterion='gini')
with mlflow.start_run(run_name='Decision Tree'):
    model.fit(X_train, y_train)

# Random Forest
model = ensemble.RandomForestClassifier(n_estimators=100, min_samples_split=2)
with mlflow.start_run(run_name='Random Forest'):
    model.fit(X_train, y_train)
    
# SVM
model = svm.SVC(C=1.0, kernel='rbf')
with mlflow.start_run(run_name='SVM'):
    model.fit(X_train, y_train)
    

<img src="images/mlflow-proper-names.png">

Notice the names are much nicer :)

You can programatically access many stored items

In [ ]:
mlflow.search_runs().columns

In [ ]:
runs = mlflow.search_runs(order_by=["metrics.training_precision_score DESC"])
runs[["tags.mlflow.runName", "metrics.training_precision_score"]]

**BTW** Each run will record the model file in the ML Flow repo. If that is starting to take up too much space, particularly if you are in the early stages and not ready for deployable models, you can turn off model loggin:

```python
mlflow.sklearn.autolog(log_models=False)     # keep params + metrics, skip the (large) model artifact
```

### Record data and metrics ML Flow doesn't know about
...such as input file version and other details

Note the use of `.log_param()`, `.log_input()` and `.log_metric()`

In [ ]:

with mlflow.start_run(run_name="rf-with-context"):
    rf = ensemble.RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

    # things autolog cannot see:
    mlflow.log_param("dataset", "titanic-numeric")
    mlflow.log_param("n_rows", len(X_train))
    mlflow.log_param("split_seed", 1)

    train_ds = mlflow.data.from_pandas(titanic_onlynum_noempty_df, 
                                       name='titanic-numeric',
                                       targets='survived')
    mlflow.log_input(train_ds, context='training')

    # an honest, held-out metric you chose (vs. the training scores in §3):
    test_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
    mlflow.log_metric("test_roc_auc", test_auc)

test_auc

In [ ]:
[m for m in dir(mlflow) if m.startswith('log_')]

### Record evaluation metrics you care about, systematically

In [ ]:
%%time

with mlflow.start_run(run_name="gbm-eval"):
    gbm = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)

    info = mlflow.sklearn.log_model(gbm, name="GradientBoosting")

    eval_df = X_test.copy()
    eval_df["survived"] = y_test.values
    result = mlflow.models.evaluate(                      
        info.model_uri, data=eval_df, targets="survived",
        model_type="classifier", evaluators="default"
    )

print("roc_auc:", round(result.metrics["roc_auc"], 3))
sorted(result.artifacts)      # everything it produced and attached to the run

In [ ]:
result.artifacts['shap_beeswarm_plot'].content

In [ ]:
result.artifacts['shap_feature_importance_plot'].content

### Detect data drift using Evidently

A natural use of Evidently is to compare the data you used to train the model against more recent model. 

In the example below, we pretend that the model was trained on passengers from first and second class and some time later, it is being used on passengers from 3rd class.

In [ ]:
#!pip install "evidently==0.7.21"

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

data = titanic_onlynum_noempty_df

behavior  = ["age", "fare", "sibsp", "parch"]
reference = data[data.pclass.isin([1, 2])][behavior]   # passengers the model 'knows'

current   = data[data.pclass == 3][behavior]           # a new batch: 3rd-class passengers

snapshot = Report([DataDriftPreset()]).run(reference_data=reference, current_data=current)
snapshot.save_html("evidently_report.html")

drift = snapshot.dict()["metrics"][0]["value"]         # {'count': n_drifted, 'share': fraction}
drift

In [ ]:
with mlflow.start_run(run_name="drift-check"):
    mlflow.log_metric("drift_share", drift["share"])
    mlflow.log_artifact("evidently_report.html")       # the report lives next to the run

View the report at ./evidently_report.html

### Check data for common errors in your data with deepcheecks

In [ ]:
# You may have to restart the kernel after running pip
#!pip install deepchecks anywidget 

In [ ]:
from deepchecks.tabular import Dataset
from deepchecks.tabular.suites import train_test_validation

ds_train = Dataset(X_train.assign(survived=y_train.values), label="survived", cat_features=[])
ds_test  = Dataset(X_test.assign(survived=y_test.values),   label="survived", cat_features=[])
result   = train_test_validation().run(ds_train, ds_test)

# with mlflow.start_run(run_name="deepchecks-validation"):
#     result.save_as_html("deepchecks_validation.html")
#     mlflow.log_artifact("deepchecks_validation.html")
#     mlflow.log_metric("deepchecks_passed",            int(result.passed(fail_if_check_not_run=False)))
#     mlflow.log_metric("deepchecks_checks_not_passed", len(result.get_not_passed_checks()))

In [ ]:
result # If this doesn't load in Jupyter Lab, try to open in VS Code

### Data drift and other checks whylogs

In [ ]:
#!pip install whylogs pybars3

In [ ]:
import whylogs as why

from whylogs.viz import NotebookProfileVisualizer

result = why.log(pandas=current) # From evidently cell
prof_view = result.view()

result_ref = why.log(pandas=reference) # From evidently cell
prof_view_ref = result_ref.view()

visualization = NotebookProfileVisualizer()
visualization.set_profiles(target_profile_view=prof_view, reference_profile_view=prof_view_ref)

visualization.summary_drift_report()
